In [2]:
import json
import os

# 🔥 MẸO CHÍ MẠNG: Tắt cơ chế kiểm tra đồng bộ số lượng <eos> của Hugging Face
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch
import numpy as np
import evaluate
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, 
    MBartForSequenceClassification, 
    Trainer, 
    TrainingArguments,
    EarlyStoppingCallback
)

# ==========================================
# 1. CẤU HÌNH ĐƯỜNG DẪN & THAM SỐ
# ==========================================
MODEL_NAME = "vinai/bartpho-word"
TRAIN_PATH = "dataset_b/train_word_filtered.json"
VAL_PATH = "dataset_b/validation_word_filtered.json"
TEST_PATH ="dataset_b/test_word_filtered.json"

OUTPUT_DIR = "output_bartpho/results_bartpho"
BEST_MODEL_DIR = "output_bartpho/best_bartpho_model"

LABEL_MAP = {"SUPPORTED": 0, "NEI": 1, "REFUTED": 2}

# ==========================================
# 2. ĐỊNH NGHĨA PYTORCH DATASET
# ==========================================
class FactCheckingDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=1024):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        claim = item["claim"]
        contexts = item["contexts"]
        
        ctx0 = contexts[0] if len(contexts) > 0 else ""
        ctx1 = contexts[1] if len(contexts) > 1 else ""
        ctx2 = contexts[2] if len(contexts) > 2 else ""
        
        sep_token = self.tokenizer.sep_token
        input_text = f"{claim} {sep_token} {ctx0} {sep_token} {ctx1} {sep_token} {ctx2}"
        
        inputs = self.tokenizer(
            input_text,
            max_length=self.max_length, 
            truncation=True,
            padding="max_length",          
            add_special_tokens=True,       
            return_tensors=None            
        )
        
        input_ids = inputs["input_ids"]
        eos_id = self.tokenizer.eos_token_id  
        pad_id = self.tokenizer.pad_token_id  
        unk_id = self.tokenizer.unk_token_id  
        
        # 🛠️ MẸO SỬA TOÁN HỌC: Đảm bảo duy nhất 1 token EOS ở cuối đoạn text
        eos_indices = [i for i, token_id in enumerate(input_ids) if token_id == eos_id]
        if len(eos_indices) > 1:
            for idx_to_fix in eos_indices[:-1]:
                input_ids[idx_to_fix] = unk_id
                
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(inputs["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(LABEL_MAP[item["label"]], dtype=torch.long)
        }

# ==========================================
# 3. HÀM ĐÁNH GIÁ (COMPUTE METRICS)
# ==========================================
metric = evaluate.load("f1")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    
    # 🛠️ SỬA LỖI MẢNG HỖN HỢP BỞI TUPLE OUTPUT CỦA BART
    if isinstance(logits, tuple):
        logits = logits[0]
        
    predictions = np.argmax(logits, axis=-1)
    macro_f1_result = metric.compute(predictions=predictions, references=labels, average="macro")
    return {"f1": macro_f1_result["f1"]}

# ==========================================
# 4. QUY TRÌNH LUỒNG CHẠY CHÍNH
# ==========================================
def load_json_data(file_path):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"❌ Không tìm thấy file dữ liệu tại: {file_path}")
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)

def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"🖥️ Thiết bị phần cứng đang sử dụng: {device.upper()}")

    print("📦 Đang nạp các tập dữ liệu JSON...")
    train_raw = load_json_data(TRAIN_PATH)
    val_raw = load_json_data(VAL_PATH)
    test_raw = load_json_data(TEST_PATH)
    
    print(f"🔹 Số lượng mẫu tập Train: {len(train_raw)}")
    print(f"🔹 Số lượng mẫu tập Validate: {len(val_raw)}")
    print(f"🔹 Số lượng mẫu tập Test: {len(test_raw)}")

    print(f"⏳ Đang cấu hình pre-trained và tokenizer: {MODEL_NAME}...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    tokenizer.padding_side = "left"
    model = MBartForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

    # 🛠️ KIỂM TRA VÀ FIX THỦ CÔNG CONFIG TOKEN ĐẶC BIỆT
    model.config.bos_token_id = tokenizer.bos_token_id
    model.config.eos_token_id = tokenizer.eos_token_id
    model.config.pad_token_id = tokenizer.pad_token_id

    train_dataset = FactCheckingDataset(train_raw, tokenizer, max_length=1024)
    val_dataset = FactCheckingDataset(val_raw, tokenizer, max_length=1024)
    test_dataset = FactCheckingDataset(test_raw, tokenizer, max_length=1024)

    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=10,                 
        per_device_train_batch_size=2,       
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,       
        gradient_checkpointing=True,         
        learning_rate=1e-5,
        weight_decay=0.01,
        warmup_ratio=0.15,
        logging_dir="./logs",
        logging_steps=10,
        eval_strategy="epoch",               
        save_strategy="epoch",               
        load_best_model_at_end=True,         
        metric_for_best_model="f1",          
        greater_is_better=True,
        fp16=torch.cuda.is_available(),     
        dataloader_num_workers=2,
        report_to="none",
        
        # 🌟 CẤU HÌNH THẦN THÁNH CHỐNG ĐẦY Ổ ĐĨA VAST.AI:
        save_only_model=True,     # 1. Chỉ lưu duy nhất file weight mô hình (~1.5GB), KHÔNG lưu optimizer rác (8GB)
        save_total_limit=3,       # 2. Chỉ giữ lại đúng 1 checkpoint tốt nhất, tự động xóa đè checkpoint cũ qua các epoch
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)] 
    )

    print("🚀 Bắt đầu quá trình huấn luyện BartPho...")
    trainer.train()

    print("\n🔍 Đang tiến hành đánh giá hiệu suất trên tập TEST độc lập...")
    test_results = trainer.evaluate(eval_dataset=test_dataset, metric_key_prefix="test")
    
    print("\n" + "="*50)
    print(" KẾT QUẢ TRÊN TẬP TEST ĐỘC LẬP")
    print("="*50)
    print(f"📊 Test Macro F1-Score: {test_results.get('test_f1', 0.0) * 100:.2f}%")
    print(f"📉 Test Loss: {test_results.get('test_loss', 0.0):.4f}")
    print("="*50)

    print(f"\n💾 Đang lưu mô hình đạt điểm F1 tối ưu vào thư mục: {BEST_MODEL_DIR}...")
    model.save_pretrained(BEST_MODEL_DIR)
    tokenizer.save_pretrained(BEST_MODEL_DIR)
    print("✅ Lưu thành công!")

if __name__ == "__main__":
    main()

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


🖥️ Thiết bị phần cứng đang sử dụng: CUDA
📦 Đang nạp các tập dữ liệu JSON...
🔹 Số lượng mẫu tập Train: 3599
🔹 Số lượng mẫu tập Validate: 750
🔹 Số lượng mẫu tập Test: 888
⏳ Đang cấu hình pre-trained và tokenizer: vinai/bartpho-word...


config.json:   0%|          | 0.00/897 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

...

Epoch,Training Loss,Validation Loss,F1
1,1.100200,1.117222,0.225751
2,1.090400,0.993722,0.485654
3,0.611000,0.866169,0.612421
4,0.682000,1.069210,0.617610
5,0.399800,0.996809,0.704494
6,0.132600,1.366918,0.702963
7,0.059900,1.750073,0.696510
8,0.004600,1.853965,0.738102
9,0.068600,2.062500,0.738820
10,0.002700,2.110048,0.733982


There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight'].



🔍 Đang tiến hành đánh giá hiệu suất trên tập TEST độc lập...


early stopping required metric_for_best_model, but did not find eval_f1 so early stopping is disabled



 KẾT QUẢ TRÊN TẬP TEST ĐỘC LẬP
📊 Test Macro F1-Score: 65.66%
📉 Test Loss: 2.6928

💾 Đang lưu mô hình đạt điểm F1 tối ưu vào thư mục: output_bartpho/best_bartpho_model...
✅ Lưu thành công!
